### Setup

In [1]:
import time
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from openai import OpenAI

### Secure API Key Loading

In [2]:
with open("../api_key.txt", "r") as f:
    API_KEY = f.read().strip()

client = OpenAI(api_key=API_KEY)


### Simple In-Memory Observability Store (Logs)

In [3]:
logs = []

### LLM Call with Logging

In [4]:
def logged_llm_call(prompt: str):
    start_time = time.time()
    success = True

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}]
        )
        output = response.choices[0].message.content

    except Exception as e:
        output = str(e)
        success = False

    latency = time.time() - start_time

    logs.append({
        "timestamp": time.time(),
        "prompt": prompt,
        "latency": latency,
        "success": success
    })

    return output

### Generate Some Traffic (Simulated Requests)

In [5]:
prompts = [
    "Explain observability in one sentence",
    "What is LLM logging?",
    "Define alerting in monitoring systems"
]

for p in prompts:
    print(logged_llm_call(p))


Observability is the ability to measure and understand the internal state of a system based on the data it generates, often through metrics, logs, and traces to diagnose issues and derive insights.
LLM logging refers to the process of recording and analyzing the interactions, outputs, and performance of large language models (LLMs) like those built on architectures such as GPT (Generative Pre-trained Transformer). This logging can encompass various types of data, including:

1. **Inputs and Outputs**: Capturing what prompts or queries are sent to the LLM, as well as the responses generated by the model. This helps in understanding how the model reacts to different stimuli.

2. **Performance Metrics**: Logging metrics related to the performance of the LLM, such as accuracy, response time, and relevance of the generated content. This can help in evaluating the effectiveness of the model in real-world applications.

3. **User Interaction Data**: Recording how users interact with the LLM, 

### Create a Real-Time Dashboard

In [6]:
# Convert Logs to DataFrame
df = pd.DataFrame(logs)

In [7]:
#Interactive Dashboard Widget
output = widgets.Output()

def update_dashboard():
    output.clear_output()
    with output:
        print("📊 LLM Observability Dashboard")
        print("Total Requests:", len(df))
        print("Failures:", (~df["success"]).sum())
        print("Avg Latency (sec):", round(df["latency"].mean(), 2))

        df["latency"].plot(kind="bar", title="Request Latency")
        plt.show()

update_dashboard()
output

Output()

### Simple Alerting Logic (Anomaly Detection)

In [8]:
LATENCY_THRESHOLD = 2.0  # seconds

def check_alerts():
    if df["latency"].max() > LATENCY_THRESHOLD:
        print("🚨 ALERT: High latency detected!")
    if (~df["success"]).any():
        print("🚨 ALERT: Request failure detected!")

check_alerts()

🚨 ALERT: High latency detected!
